# Financial Risk Intelligence using Sentence Embeddings, Logistic Regression and Retrieval-Augmented Generation

**Course:** 42.515 Machine Learning and Analytics  
**Semester/Year:** Spring 2026

---

## 1. Package Installation & Imports

All libraries required to run this project are installed and imported here.
- `sentence-transformers` — converts text into numerical embeddings
- `scikit-learn` — trains and evaluates the risk classifier
- `transformers` — loads the local LLM for explanation generation
- `pandas / numpy` — data loading and manipulation

In [1]:
# Install required packages (run once)
!pip install sentence-transformers transformers scikit-learn pandas numpy

import pandas as pd
import numpy as np
import re

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModelForCausalLM, AutoTokenizer

print("All packages imported successfully!")


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


All packages imported successfully!


## 2. Load Embedding Model

We use `all-MiniLM-L6-v2` from SentenceTransformers.  
This model converts any sentence into a 384-dimensional vector that captures its meaning.  
Sentences with similar meaning will have vectors that are close together in this space.

In [2]:
# Load the sentence embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded!


## 3. Load News Dataset (Retriever Corpus)

`cleaned_news_dataset.csv` contains **35,191** financial news articles with three columns:
- `text1` — the article headline
- `Description` — the full article description
- `Time` — publication timestamp

This dataset is used exclusively by the **Retriever Agent** to find relevant articles for a query.  
Place the CSV file in the same folder as this notebook before running.

In [ ]:
# Load news dataset — place CSV in the same folder as this notebook
#dataset link: https://www.kaggle.com/datasets/notlucasp/financial-news-headlines
news_df = pd.read_csv(r"D:\Machine Learning\Project\Group_8\Datasets\cleaned_news_dataset.csv")

# Use Description as retrieval text; drop empty rows
news_df["retrieval_text"] = news_df["Description"].astype(str)
news_df = news_df[news_df["retrieval_text"].str.strip() != ""].reset_index(drop=True)

print(f"News dataset loaded: {len(news_df):,} articles")
print(f"Columns: {news_df.columns.tolist()}")
news_df[["text1", "Description"]].head(3)

News dataset loaded: 35,191 articles
Columns: ['text1', 'Time', 'Description', 'retrieval_text']


,text1,Description
0,Jim Cramer A better way to invest in the Covid...,"""mad money"" host jim cramer recommended buying..."
1,Cramers lightning round I would own Teradyne,"""mad money"" host jim cramer rings the lightnin..."
2,"Cramers week ahead Big week for earnings, even...","""we'll pay more for the earnings of the non-co..."


## 4. Load Sentiment Dataset (Classifier Training Data)

`cleaned_sentiment_data.csv` contains **4,149** labelled financial sentences with columns:
- `text` — the financial sentence
- `Sentiment` — label: `positive`, `neutral`, or `negative`

Label distribution: **neutral 2,083 | positive 1,555 | negative 511**

This dataset is used exclusively by the **Analyzer Agent** to train the risk classifier.

In [ ]:
# Load sentiment dataset — place CSV in the same folder as this notebook
#dataset link: https://huggingface.co/datasets/chiapudding/kaggle-financial-sentiment
sent_df = pd.read_csv(r"D:\Machine Learning\Project\Group_8\Datasets\cleaned_sentiment_data.csv")
sent_df = sent_df[["text", "Sentiment"]].dropna()

print(f"Sentiment dataset loaded: {len(sent_df):,} rows")
print("\nSentiment distribution:")
print(sent_df["Sentiment"].value_counts())
sent_df.head(3)

Sentiment dataset loaded: 4,149 rows

Sentiment distribution:
Sentiment
neutral     2083
positive    1555
negative     511
Name: count, dtype: int64


,text,Sentiment
0,geosolution benefon 's,positive
1,esi 1.50 2.50,negative
2,componenta 's eur131 eur76 eur7,positive


## 5. Create Risk Labels

We convert the three sentiment labels into financial risk levels:

| Sentiment | Risk Level  |
|-----------|-------------|
| negative  | High Risk   |
| neutral   | Medium Risk |
| positive  | Low Risk    |

This becomes the target variable `y` for the ML classifier.

In [5]:
def map_risk(sentiment: str) -> str:
    """Map sentiment label to a financial risk level."""
    s = sentiment.strip().lower()
    if s == "negative":
        return "High Risk"
    elif s == "neutral":
        return "Medium Risk"
    else:
        return "Low Risk"

sent_df["risk"] = sent_df["Sentiment"].apply(map_risk)

print("Risk label distribution:")
print(sent_df["risk"].value_counts())

Risk label distribution:
risk
Medium Risk    2083
Low Risk       1555
High Risk       511
Name: count, dtype: int64


## 6. Agent 2 — Analyzer Agent (ML Risk Classifier)

**Goal:** Predict the financial risk level of any news text.

**How it works:**
1. Each sentence in `sent_df` is encoded into a 384-dim embedding vector
2. A Logistic Regression model is trained on these vectors with `risk` as the target
3. At inference time, any new text is embedded and passed to the trained model

`class_weight="balanced"` is used to compensate for the class imbalance  
(negative/High Risk has only 511 samples vs 2,083 neutral).

In [6]:
# Split into train / test sets
X_train, X_test, y_train, y_test = train_test_split(
    sent_df["text"], sent_df["risk"],
    test_size=0.2, random_state=42, stratify=sent_df["risk"]
)

print("Encoding training data...")
X_train_emb = embedding_model.encode(X_train.tolist(), show_progress_bar=True)
X_test_emb  = embedding_model.encode(X_test.tolist(),  show_progress_bar=True)

# Train Logistic Regression classifier
clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
clf.fit(X_train_emb, y_train)

# Evaluate
y_pred = clf.predict(X_test_emb)
print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Encoding training data...


Batches:   0%|          | 0/104 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]


Accuracy: 0.5108

Classification Report:
              precision    recall  f1-score   support

   High Risk       0.24      0.53      0.33       102
    Low Risk       0.49      0.39      0.43       311
 Medium Risk       0.70      0.60      0.64       417

    accuracy                           0.51       830
   macro avg       0.48      0.51      0.47       830
weighted avg       0.56      0.51      0.53       830



### Analyzer Helper Function

A convenience wrapper that encodes any text string and returns its predicted risk level.

In [7]:
def predict_risk(text: str) -> str:
    """Predict financial risk level for a given text string."""
    emb = embedding_model.encode([text])
    return clf.predict(emb)[0]

# Sanity checks
print(predict_risk("The company reported a massive drop in quarterly profits."))
print(predict_risk("Revenue grew steadily, beating analyst expectations."))

High Risk
Low Risk


## 7. Agent 1 — Retriever Agent (Semantic Search)

**Goal:** Given a user query, find the most relevant articles from `cleaned_news_dataset.csv`.

**How it works:**
1. All 35,191 news descriptions are encoded into embedding vectors once at startup
2. The user query is encoded into a vector
3. Cosine similarity is computed between the query vector and every news vector
4. The top-K highest-similarity articles are returned

Cosine similarity between two vectors **u** and **v** is:

$$\cos(\theta) = \frac{u \cdot v}{\|u\| \|v\|}$$

A value of 1 means identical direction (very similar meaning); 0 means unrelated.

In [8]:
# Pre-compute embeddings for the full news corpus (run once)
print(f"Encoding {len(news_df):,} news articles for retrieval...")
news_texts      = news_df["retrieval_text"].tolist()
news_embeddings = embedding_model.encode(news_texts, batch_size=32, show_progress_bar=True)
print("Retriever ready!")

def retrieve_news(query: str, top_k: int = 3) -> list:
    """
    Retrieve top-K most semantically similar news articles for a given query.
    Uses cosine similarity between query embedding and pre-computed news embeddings.
    """
    query_emb = embedding_model.encode([query])
    sim       = cosine_similarity(query_emb, news_embeddings)[0]
    top_idx   = sim.argsort()[-top_k:][::-1]

    results = []
    for idx in top_idx:
        row = news_df.iloc[idx]
        results.append({
            "headline"   : str(row["text1"])[:100],
            "description": str(row["Description"]),
            "score"      : float(sim[idx])
        })
    return results

# Quick test
sample = retrieve_news("company facing financial losses", top_k=2)
for s in sample:
    print(f"[{s['score']:.3f}] {s['headline']}")

Encoding 35,191 news articles for retrieval...


Batches:   0%|          | 0/1100 [00:00<?, ?it/s]

Retriever ready!
[0.558] MUFG reports first quarterly loss in decade on writedown at Indonesian unit
[0.558] AIG swings to loss, hit by catastrophes and volatile market


## 8. Agent 3 — Generator Agent (LLM Explanation)

**Goal:** Produce a plain-English explanation of the predicted risk level.

We use **Qwen3-0.6B**, a lightweight open-source LLM that runs locally.  
The model receives the headline, description, and predicted risk as context,  
then generates a 2–3 sentence explanation for a non-technical reader.

In [9]:
# Load local LLM
model_name = "Qwen/Qwen3-0.6B"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
llm_model  = AutoModelForCausalLM.from_pretrained(model_name)
print("LLM loaded!")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

LLM loaded!


### Generator Function

**Bug fix:** The original code decoded the full output including the prompt,  
causing the prompt text to appear in every response.  
We now slice off exactly `prompt_len` tokens before decoding,  
so only the newly generated explanation is returned.

In [10]:
def generate_explanation(headline: str, description: str, risk: str) -> str:
    """
    Generate a plain-English explanation for the predicted risk level.
    Only the newly generated tokens are decoded (prompt tokens are sliced off).
    """
    prompt = (
        "You are a financial analyst. Read the news below and explain "
        f"in 2-3 simple sentences why this is classified as {risk}.\n\n"
        f"Headline: {headline}\n"
        f"Description: {description}\n\n"
        "Explanation:"
    )

    inputs     = tokenizer(prompt, return_tensors="pt")
    prompt_len = inputs["input_ids"].shape[1]        # number of prompt tokens

    output = llm_model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,                             # greedy decoding = deterministic
        pad_token_id=tokenizer.eos_token_id
    )

    # KEY FIX: slice off the prompt tokens, decode only the new explanation tokens
    new_tokens  = output[0][prompt_len:]
    explanation = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return explanation

## 9. Agent 4 — Critic Agent (Quality Validation)

**Goal:** Validate the generated explanation before showing it to the user.

**Three checks performed:**
1. **Length check** — explanation must be at least 10 words
2. **Consistency check** — the risk keyword (e.g. "high", "low") must appear in the explanation
3. **Repetition check** — explanation must not be a near-verbatim copy of the input

If any check fails, the Critic substitutes a safe fallback explanation  
rather than returning something misleading or empty.

In [11]:
def critic_agent(explanation: str, risk: str, description: str) -> dict:
    """
    Validate the generated explanation for quality and consistency.
    Returns a dict with: passed (bool), issues (list), explanation (str).
    """
    issues   = []
    risk_kw  = risk.lower().split()[0]   # "high", "medium", or "low"

    # Check 1: minimum length
    word_count = len(explanation.split())
    if word_count < 10:
        issues.append(f"Too short ({word_count} words; minimum is 10)")

    # Check 2: risk keyword present
    if risk_kw not in explanation.lower():
        issues.append(f"Risk keyword '{risk_kw}' not mentioned in explanation")

    # Check 3: not repeating input verbatim (overlap > 80% = likely echo)
    desc_words = set(re.findall(r'\w+', description.lower()))
    expl_words = set(re.findall(r'\w+', explanation.lower()))
    if expl_words and desc_words:
        overlap = len(desc_words & expl_words) / len(expl_words)
        if overlap > 0.8:
            issues.append("Explanation appears to repeat the input description verbatim")

    passed = len(issues) == 0
    final_explanation = (
        explanation if passed
        else f"[Auto-summary] This article is classified as {risk} based on its detected financial sentiment."
    )

    return {"passed": passed, "issues": issues, "explanation": final_explanation}

# Quick test
res = critic_agent("This company faces high risk due to declining profits.", "High Risk", "profits dropped")
print("Passed:", res["passed"])
print("Output:", res["explanation"])

Passed: False
Output: [Auto-summary] This article is classified as High Risk based on its detected financial sentiment.


## 10. Full Pipeline — `run_system()`

All four agents are wired together here:

```
User Query
    ↓
[Agent 1: Retriever]  — finds top-K similar articles from cleaned_news_dataset.csv
    ↓
[Agent 2: Analyzer]   — predicts risk level using model trained on cleaned_sentiment_data.csv
    ↓
[Agent 3: Generator]  — explains the risk in plain English using Qwen3-0.6B
    ↓
[Agent 4: Critic]     — validates the explanation for quality and consistency
    ↓
Final output to user
```

In [12]:
def run_system(query: str, top_k: int = 3) -> list:
    """
    Full multi-agent pipeline.
    Returns a list of result dicts (one per retrieved article).
    """
    # Agent 1: Retrieve relevant news from cleaned_news_dataset.csv
    retrieved = retrieve_news(query, top_k=top_k)

    final_results = []
    for item in retrieved:

        # Agent 2: Predict risk using model trained on cleaned_sentiment_data.csv
        risk = predict_risk(item["description"])

        # Agent 3: Generate plain-English explanation
        explanation = generate_explanation(item["headline"], item["description"], risk)

        # Agent 4: Validate the explanation
        critique = critic_agent(explanation, risk, item["description"])

        final_results.append({
            "headline"   : item["headline"],
            "similarity" : round(item["score"], 3),
            "risk"       : risk,
            "explanation": critique["explanation"],
            "qa_passed"  : critique["passed"],
            "qa_issues"  : critique["issues"]
        })

    return final_results

## 11. Test the System

Run a sample query end-to-end through all four agents and display the results.

In [13]:
query   = "Which companies are facing financial losses?"
results = run_system(query, top_k=3)

for i, r in enumerate(results, 1):
    print(f"\n{'='*60}")
    print(f"Result      {i}")
    print(f"Headline  : {r['headline']}")
    print(f"Similarity: {r['similarity']}")
    print(f"Risk Level: {r['risk']}")
    print(f"QA Passed : {r['qa_passed']}")
    if r['qa_issues']:
        print(f"QA Issues : {', '.join(r['qa_issues'])}")
    print(f"Explanation: {r['explanation']}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Result      1
Headline  : Fasten your seatbelt Investors brace for Europe Inc. results amid coronavirus
Similarity: 0.591
Risk Level: Low Risk
QA Passed : False
QA Issues : Risk keyword 'low' not mentioned in explanation
Explanation: [Auto-summary] This article is classified as Low Risk based on its detected financial sentiment.

Result      2
Headline  : Big U.S. banks predict more economic pain from coronavirus
Similarity: 0.566
Risk Level: High Risk
QA Passed : False
QA Issues : Risk keyword 'high' not mentioned in explanation
Explanation: [Auto-summary] This article is classified as High Risk based on its detected financial sentiment.

Result      3
Headline  : Miners face funding squeeze as green investing surges
Similarity: 0.557
Risk Level: Low Risk
QA Passed : True
Explanation: The mining companies are facing a significant funding shortage due to increased demand for green investments, which is a positive trend in the market. This shift in investment direction is expected to r